In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import scanpy as sc
import diopy
import scrublet as scr
import matplotlib.pyplot as plt
import scanpy.external as sce
import sctour as sct
import scipy.sparse

sc.settings.set_figure_params(dpi=1000,figsize=(5, 5))
sc.logging.print_header()

In [ ]:
## data dir
origin_dir = " "
input_dir = " "
output_dir = " "

## set
random_state = 123
nepoch = 20

## color
tumor_status_color = {"Tumor":"#5BC0EB","Metastasis":"#8DD3C7","NAT":"#9BC53D","Healthy":"#C3423F"}

In [ ]:
## data read
scRNA_origin = diopy.input.read_h5(file = f"{origin_dir}/scRNA_gene_filter.h5")
scRNA_tumor_cell = sc.read_h5ad(f"{input_dir}/scRNA_cell_type_cluster.h5ad")
# scRNA_tumor_cell = diopy.input.read_h5(file = f"{input_dir}/scRNA_cell_type_cluster_only.h5")

In [ ]:
## hub info copy
scRNA_origin.obs = scRNA_tumor_cell.obs.copy()
scRNA_origin.obsm = scRNA_tumor_cell.obsm.copy()

In [ ]:
## pre process
scRNA_origin.layers["counts"] = scRNA_origin.X.copy()
sc.pp.normalize_total(scRNA_origin, target_sum=1e4)
sc.pp.log1p(scRNA_origin)
sc.pp.calculate_qc_metrics(scRNA_origin, percent_top=None, log1p=False, inplace=True)
sc.pp.highly_variable_genes(scRNA_origin, flavor='seurat',batch_key= "tumor_code", n_top_genes=5000, subset=True)
# sc.pp.highly_variable_genes(scRNA_origin, flavor='seurat_v3', n_top_genes=5000, subset=True)

In [ ]:
# if scipy.sparse.issparse(scRNA_origin.X):
#     scRNA_origin.X = scRNA_origin.X.toarray()
# else:
#     scRNA_origin.X = np.array(scRNA_origin.X)
# scRNA_origin.X = scRNA_origin.X.astype(np.float32)

scRNA_origin.X = scRNA_origin.layers["counts"].copy()

In [ ]:
## scTour run
tnode = sct.train.Trainer(scRNA_origin, loss_mode='nb', alpha_recon_lec=0.5, alpha_recon_lode=0.5, nepoch=nepoch, 
                          use_gpu=False, random_state = random_state)
tnode.train()
scRNA_origin.obs['ptime'] = tnode.get_time()
mix_zs, zs, pred_zs = tnode.get_latentsp(alpha_z=0.5, alpha_predz=0.5)
scRNA_origin.obsm['X_TNODE'] = mix_zs
scRNA_origin.obsm['X_VF'] = tnode.get_vector_field(scRNA_origin.obs['ptime'].values, scRNA_origin.obsm['X_TNODE'])
V = scRNA_origin.obsm["X_VF"]
scRNA_origin.obsm["X_UMAP_RAW_FOR_VF"] = scRNA_origin.obsm["X_umap"].copy()
scRNA_origin.obsm["X_VF_2D"] = V[:, :2].copy()

In [ ]:
scRNA_origin

In [ ]:
## new umap
scRNA_origin=scRNA_origin[np.argsort(scRNA_origin.obs['ptime'].values), :]

sc.pp.neighbors(scRNA_origin, use_rep='X_TNODE', n_neighbors=15)
sc.tl.umap(scRNA_origin, min_dist=0.1)


In [ ]:
## result plot
sct.vf.plot_vector_field(scRNA_origin,zs_key="X_UMAP_RAW_FOR_VF",  # 指定2D坐标（原始UMAP）
                         vf_key="X_VF_2D",  # 指定2D向量场数据
                         use_rep_neigh="X_TNODE",  # 指定用于构建邻居图的潜空间特征
                         color="tumor_status",  # 按细胞类型着色
                         show=False,  # 不直接显示图像（后续统一调整显示）
                        #  ax=ax, # 指定绘图的子图对象
                         legend_loc="none",  # 不显示图例
                         frameon=False,  # 不显示图像边框
                         size=5,  # 设置点的大小
                         alpha=0.2, # 设置点的透明度（避免点重叠遮挡）
                         palette = tumor_status_color
                         )
plt.savefig(f'{output_dir}/scRNA_tumor_cell_vector_field_{nepoch}.png', dpi=1500)

In [ ]:
sc.pl.umap(scRNA_origin, color='ptime', show=False, frameon=False)
plt.savefig(f'{output_dir}/scRNA_tumor_cell_ptime_{nepoch}_legend.png', dpi=1500)
sc.pl.umap(scRNA_origin, color='ptime', show=False, frameon=False,legend_loc="none")
plt.savefig(f'{output_dir}/scRNA_tumor_cell_ptime_{nepoch}_nolegend.png', dpi=1500)

In [ ]:
## data save
scRNA_origin.obs.reset_index().to_csv(f"{output_dir}/scRNA_tumor_cell_metadata_{nepoch}.csv", index=False)
scRNA_origin.write_h5ad(f"{output_dir}/scRNA_tumor_cell_scTour_{nepoch}.h5ad", compression="gzip")